# Week 1 · Feature engineering and causal preprocessing

This notebook prepares the campaign data for causal-effect modeling. It checks the raw file, fixes only verified issues, makes a few useful pre-campaign features, keeps the treatment and outcomes separate from customer features, and creates a fair train/test split.

**Important:** the supplied Hillstrom file records email groups, not $10/$20 discounts. This notebook prepares the email experiment; it cannot estimate discount effects that the dataset never recorded.

Run from the project folder in VS Code. The notebook finds `data/raw/hillstrom.csv` by searching the current folder and its parents. It writes prepared outputs under `data/processed/` and a fitted encoder under `models/`.

In [ ]:
from pathlib import Path
import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

SEED = 42
ARMS = {"No E-Mail": 0, "Mens E-Mail": 1, "Womens E-Mail": 2}
OUTCOMES = ["visit", "conversion", "spend"]

# Find the repository folder even if VS Code starts the notebook one level below it.
def find_project_root(start=Path.cwd()):
    for folder in [start.resolve(), *start.resolve().parents]:
        if (folder / "data" / "raw" / "hillstrom.csv").is_file():
            return folder
    raise FileNotFoundError("Could not find data/raw/hillstrom.csv. Open the project folder in VS Code.")

ROOT = find_project_root()
RAW_PATH = ROOT / "data" / "raw" / "hillstrom.csv"
print(f"Project folder: {ROOT}")
print(f"Raw data file:  {RAW_PATH}")

## 1. Look at the source before changing it

First inspect its size, columns, missing values, repeated rows, group names and basic number ranges. Repeated-looking rows are reported, not automatically deleted: the file has no real customer ID to prove that they are accidental copies.

In [ ]:
raw = pd.read_csv(RAW_PATH)
required_columns = {
    "recency", "history_segment", "history", "mens", "womens", "zip_code",
    "newbie", "channel", "segment", "visit", "conversion", "spend"
}
missing_columns = sorted(required_columns - set(raw.columns))
if missing_columns:
    raise ValueError(f"The source file is missing required columns: {missing_columns}")
if raw.empty:
    raise ValueError("The source file has no rows.")

print(f"Rows: {len(raw):,} | Columns: {raw.shape[1]}")
print(f"Missing cells: {int(raw.isna().sum().sum()):,}")
print(f"Rows identical across all source columns: {int(raw.duplicated().sum()):,}")
print("\nEmail groups:")
print(raw["segment"].value_counts(dropna=False).to_string())
print("\nData types:")
print(raw.dtypes.to_string())
print("\nNumber ranges:")
print(raw[["recency", "history", "mens", "womens", "newbie", "visit", "conversion", "spend"]].describe().round(2).to_string())
raw.head()

## 2. Validate and clean without guessing

The checks below stop the notebook if the file has missing or impossible values. The one known spelling issue is `Surburban`, which is corrected to `Suburban`. No customer is removed, no missing value is filled with a guess, and unusually high but valid spending is retained.

In [ ]:
data = raw.copy()
if data[list(required_columns)].isna().any().any():
    bad = data[list(required_columns)].isna().sum()
    raise ValueError(f"Required fields contain missing values:\n{bad[bad > 0]}")

numeric_columns = ["recency", "history", "mens", "womens", "newbie", "visit", "conversion", "spend"]
for column in numeric_columns:
    data[column] = pd.to_numeric(data[column], errors="raise")
    if not np.isfinite(data[column]).all():
        raise ValueError(f"{column} contains a non-finite number.")

for column in ["mens", "womens", "newbie", "visit", "conversion"]:
    if not data[column].isin([0, 1]).all():
        raise ValueError(f"{column} must contain only 0 or 1.")
if not data["recency"].between(1, 12).all() or (data["recency"] % 1 != 0).any():
    raise ValueError("recency must be a whole number from 1 to 12.")
if (data["history"] < 0).any() or (data["spend"] < 0).any():
    raise ValueError("Past history and campaign spend cannot be negative.")
if not data["segment"].isin(ARMS).all():
    raise ValueError(f"Unexpected campaign groups: {sorted(set(data['segment']) - set(ARMS))}")
if (data["conversion"] > data["visit"]).any():
    raise ValueError("A purchase is recorded for a customer without a visit.")
if ((data["spend"] > 0) != (data["conversion"] == 1)).any():
    raise ValueError("Positive spend and purchase indicators do not agree.")

# Fix the one known typo. Keep all other category values as they are.
print("ZIP-code labels before correction:", sorted(data["zip_code"].unique()))
data["zip_code"] = data["zip_code"].replace({"Surburban": "Suburban"})
print("ZIP-code labels after correction: ", sorted(data["zip_code"].unique()))

# The source has no true customer identifier. This stable row label is only for joining files.
data.insert(0, "customer_id", [f"row_{i:06d}" for i in range(1, len(data) + 1)])
data["treatment"] = data["segment"].map(ARMS).astype("int8")
print(f"\nRows kept: {len(data):,} of {len(raw):,}")
print("Treatment codes:", ARMS)

## 3. Make a small set of useful pre-campaign features

These features use information known before the email was sent:

- `history_log1p` softens the long right tail of past spending, so very large shoppers do not dominate a simple distance-based comparison.
- `recency_band` groups recency into 1–4, 5–8 and 9–12 for an easier-to-read recency pattern. The original numeric `recency` is kept too.
- `prior_interest_count` counts whether the customer previously bought from zero, one or both product categories. The original `mens` and `womens` flags are kept so the model can distinguish the two interests.

`history_segment` is left out because it is just a bucketed version of `history`. The email group and later outcomes (`visit`, `conversion`, `spend`) are kept as separate treatment/target columns and never enter the customer-feature matrix. This prevents target leakage.

In [ ]:
data["history_log1p"] = np.log1p(data["history"])
data["recency_band"] = pd.cut(
    data["recency"],
    bins=[0, 4, 8, 12],
    labels=["recent_1_4", "middle_5_8", "longer_ago_9_12"],
    include_lowest=True,
).astype("string")
data["prior_interest_count"] = (data["mens"] + data["womens"]).astype("int8")

FEATURES = [
    "recency", "history", "history_log1p", "mens", "womens",
    "prior_interest_count", "newbie", "recency_band", "zip_code", "channel"
]
CATEGORICAL_FEATURES = ["recency_band", "zip_code", "channel"]
NUMERIC_FEATURES = [name for name in FEATURES if name not in CATEGORICAL_FEATURES]

# Show why the log feature may help, while keeping the original dollars available too.
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].hist(data["history"], bins=40, color="#3977a8")
axes[0].set_title("Past spend (original)")
axes[0].set_xlabel("History")
axes[1].hist(data["history_log1p"], bins=40, color="#4b9b79")
axes[1].set_title("Past spend (log1p feature)")
axes[1].set_xlabel("log(1 + history)")
plt.tight_layout()
plt.show()

data[["history", "history_log1p", "recency", "recency_band", "mens", "womens", "prior_interest_count"]].head()

## 4. Keep treatment, outcomes and features in their proper roles

The treatment is coded `0 = no email`, `1 = men's email`, `2 = women's email`. The primary outcome for the project is `visit`; `conversion` and `spend` are additional outcomes. The project can compare email offers with no email because those assignments exist in this data. It cannot compare discount amounts without discount records.

For this randomized study, baseline features help the model learn which customer profiles respond differently. They are not automatically confounders: random assignment means the pre-campaign features did not decide who got each email.

In [ ]:
# Keep the causal inputs visibly separate.
X = data[FEATURES].copy()                  # customer information known before treatment
T = data["treatment"].copy()               # email assignment (0, 1, or 2)
Y = data[OUTCOMES].copy()                   # outcomes measured after assignment

forbidden_from_X = {"treatment", "segment", *OUTCOMES, "history_segment", "customer_id"}
assert not (set(FEATURES) & forbidden_from_X), "Treatment/outcome leakage found in X."
assert X.notna().all().all() and Y.notna().all().all()

print("Feature columns:", FEATURES)
print("\nTreatment counts:")
print(T.map({v: k for k, v in ARMS.items()}).value_counts().to_string())
print("\nMean outcomes by randomized email group (descriptive check):")
print(data.groupby("segment")[OUTCOMES].mean().round(4).to_string())

## 5. Split once; learn category encoding from training rows only

We keep 70% for training and 30% for a final holdout test. Stratifying by treatment keeps each email group represented in both parts. The test outcomes are not used to fit the encoder. Forest models do not need feature scaling, so numeric values are passed through as they are.

In [ ]:
train_ids, test_ids = train_test_split(
    np.arange(len(data)),
    test_size=0.30,
    random_state=SEED,
    stratify=T,
)
train = data.iloc[train_ids].reset_index(drop=True)
test = data.iloc[test_ids].reset_index(drop=True)

preprocessor = ColumnTransformer(
    transformers=[
        ("numbers", "passthrough", NUMERIC_FEATURES),
        ("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)
preprocessor.fit(train[FEATURES])
feature_names = list(preprocessor.get_feature_names_out())
X_train = pd.DataFrame(preprocessor.transform(train[FEATURES]), columns=feature_names)
X_test = pd.DataFrame(preprocessor.transform(test[FEATURES]), columns=feature_names)

# Make sure every row is used once, the two splits do not overlap, and categories are numeric now.
assert len(train) + len(test) == len(data)
assert set(train.customer_id).isdisjoint(set(test.customer_id))
assert set(train.treatment.unique()) == set(test.treatment.unique()) == {0, 1, 2}
assert X_train.columns.tolist() == X_test.columns.tolist()
print(f"Training rows: {len(train):,} | Test rows: {len(test):,}")
print(f"Encoded feature count: {len(feature_names)}")
print("Training group shares:", train.segment.value_counts(normalize=True).round(3).to_dict())
print("Test group shares:    ", test.segment.value_counts(normalize=True).round(3).to_dict())
print("Encoded columns:", feature_names)

## 6. Save files for the next modeling step

The clean file retains the engineered customer information plus the separate treatment and outcomes. The train/test files contain one-hot encoded `X` columns, a treatment code, a human-readable treatment name, and outcomes. The fitted preprocessor is saved so future customer data can be transformed using the same training categories.

In [ ]:
processed_dir = ROOT / "data" / "processed"
models_dir = ROOT / "models"
results_dir = ROOT / "results"
for folder in [processed_dir, models_dir, results_dir]:
    folder.mkdir(parents=True, exist_ok=True)

train_ready = X_train.copy()
train_ready.insert(0, "customer_id", train["customer_id"].to_numpy())
train_ready["treatment"] = train["treatment"].to_numpy()
train_ready["treatment_name"] = train["segment"].to_numpy()
for outcome in OUTCOMES:
    train_ready[outcome] = train[outcome].to_numpy()

test_ready = X_test.copy()
test_ready.insert(0, "customer_id", test["customer_id"].to_numpy())
test_ready["treatment"] = test["treatment"].to_numpy()
test_ready["treatment_name"] = test["segment"].to_numpy()
for outcome in OUTCOMES:
    test_ready[outcome] = test[outcome].to_numpy()

# Save outputs with new names so existing project files are not overwritten.
data.to_csv(processed_dir / "causal_clean.csv", index=False)
train_ready.to_csv(processed_dir / "causal_train.csv", index=False)
test_ready.to_csv(processed_dir / "causal_test.csv", index=False)
joblib.dump(preprocessor, models_dir / "causal_preprocessor.joblib")

summary = {
    "source": str(RAW_PATH.relative_to(ROOT)),
    "rows_kept": int(len(data)),
    "source_duplicate_looking_rows_retained": int(raw.duplicated().sum()),
    "treatment_codes": ARMS,
    "feature_columns_before_encoding": FEATURES,
    "feature_columns_after_encoding": feature_names,
    "outcomes": OUTCOMES,
    "train_rows": int(len(train)),
    "test_rows": int(len(test)),
    "seed": SEED,
    "decisions": [
        "No rows were deleted or imputed; duplicate-looking rows were retained because no true customer ID exists.",
        "Corrected the known zip label typo Surburban to Suburban.",
        "Added log1p(history), recency bands, and count of prior product interests.",
        "Excluded treatment and outcomes from X to prevent leakage.",
        "Fit one-hot categories on training rows only; test-only categories are ignored safely.",
        "This dataset supports email-treatment effects, not discount-amount effects."
    ],
}
with (results_dir / "causal_preprocessing_summary.json").open("w") as file:
    json.dump(summary, file, indent=2)

print("Saved:")
for path in [processed_dir / "causal_clean.csv", processed_dir / "causal_train.csv", processed_dir / "causal_test.csv", models_dir / "causal_preprocessor.joblib", results_dir / "causal_preprocessing_summary.json"]:
    print(" -", path.relative_to(ROOT))
print("\nTrain-ready preview:")
train_ready.head()

## What this preprocessing does and does not do

It makes the input consistent, creates useful information from pre-campaign fields, and prevents the model from accidentally seeing treatment or future outcomes as customer features. It does not prove causality by itself; random assignment, the causal graph, the Double ML model, and the held-out evaluation are the later steps.

The generated `customer_id` values identify rows only. They are not known real-world customer IDs. Purchase and spend remain rare, so we keep their original values and do not create synthetic purchases or rebalance the outcomes.